In [1]:

import pandas as pd
from wearable_affect.data import PROJECT_ROOT
from wearable_affect.evaluation import loso_evaluate, summarise
from wearable_affect.models import make_dummy, make_logreg, make_lightgbm

features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "features.parquet")
print(features.shape)
print(features.groupby("target")[["scl_mean", "scr_count", "hr_mean"]].mean())
print(features[["hr_mean", "hrv_sdnn", "hrv_rmssd"]].describe().T)

(1015, 17)
        scl_mean  scr_count    hr_mean
target                                
0       1.339609   7.249649  75.696366
1       3.439927  19.870861  88.704947
            count        mean         std        min         25%         50%  \
hr_mean    1015.0   79.566900   11.227760  55.631848   72.510533   78.046512   
hrv_sdnn   1015.0  273.550038  298.530682  36.681316  168.113387  223.704555   
hrv_rmssd  1015.0  365.273441  409.705654  28.894648  227.896178  302.398034   

                  75%          max  
hr_mean     84.918366   140.909141  
hrv_sdnn   285.927626  4923.734937  
hrv_rmssd  381.698599  7186.770760  


In [2]:
results = {}
for name, make_model in [("dummy", make_dummy), ("logreg", make_logreg), ("lightgbm", make_lightgbm)]:
    results[name] = loso_evaluate(features, make_model)
    print(f"\n{name}")
    print(summarise(results[name]).round(3))


dummy
                    mean    std
macro_f1           0.413  0.004
balanced_accuracy  0.500  0.000
auroc              0.500  0.000

logreg
                    mean    std
macro_f1           0.831  0.112
balanced_accuracy  0.852  0.097
auroc              0.966  0.047

lightgbm
                    mean    std
macro_f1           0.850  0.144
balanced_accuracy  0.866  0.120
auroc              0.976  0.038


In [3]:
per_subject = pd.DataFrame({name: r.set_index("subject_id")["macro_f1"] for name, r in results.items()})
per_subject.round(3)

,dummy,logreg,lightgbm
subject_id,,,
S10,0.405,0.891,0.736
S11,0.412,0.966,0.890
S13,0.415,0.636,1.000
S14,0.415,0.894,0.415
S15,0.410,0.650,0.930
S16,0.412,0.811,0.982
S17,0.405,0.902,0.820
S2,0.414,0.678,0.852
S3,0.412,0.791,0.735
